In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_customers = spark.table("medalhao.bronze.tb_customers") #data frame das tabelas do bronze
df_geolocalizacao = spark.table("medalhao.bronze.tb_geolocalizacao")
df_items = spark.table("medalhao.bronze.tb_items")
df_orders = spark.table("medalhao.bronze.tb_orders")
df_order_items = spark.table("medalhao.bronze.tb_order_items")
df_order_payments = spark.table("medalhao.bronze.tb_order_payments")
df_order_reviews = spark.table("medalhao.bronze.tb_order_reviews")
df_products = spark.table("medalhao.bronze.tb_products")
df_sellers = spark.table("medalhao.bronze.tb_sellers")
df_product_category_name_translation = spark.table("medalhao.bronze.tb_product_category_name_translation")
df_cotacao = spark.table("medalhao.bronze.tb_cotacao")

catalogo = "medalhao"
silver_db_name = "silver"



In [0]:
df_customers.display()

In [0]:
silver_dim_consumidores = (df_customers
                          .withColumnRenamed("customer_id", "id_consumidor")
                          .withColumnRenamed("customer_zip_code_prefix", "prefixo_cep")
                          .withColumnRenamed("customer_city", "cidade")
                          .withColumnRenamed("customer_state", "estado")
                          .withColumnRenamed("customer_name", "nome_consumidor"))

silver_dim_consumidores = (silver_dim_consumidores.withColumn("cidade", F.upper(F.col("cidade"))) #aplicando a regra de negócio
                           .withColumn("estado", F.upper(F.col("estado"))))

janela_deduplicacao = Window.partitionBy("id_consumidor").orderBy(F.col("timestamp_ingestion"))

silver_dim_consumidores = (silver_dim_consumidores
                           .withColumn("rn", F.rank().over(janela_deduplicacao))
                           .filter(F.col("rn") == 1)
                           .drop("rn"))
(silver_dim_consumidores
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{catalogo}.{silver_db_name}.dim_consumidores"))


silver_dim_consumidores.display()
